# 2026-09-02

## Topics

+ Measuring and handling "blockages" in the telescope/camera

## Discuss: Imagine you took an image of a perfectly uniform source (e.g. small piece of the lit-up sky, or a smooth screen with white light on it. What should the image look like?




## Some image comparisons

|  |  |
| -- | -- |
| **Single flat, V-band** | **Single flat, B-band** |
| ![Single flat, V-band](media/flat-V-single-2026-08-22.png) | ![Single flat, B-band](media/flat-B-single-2026-08-22.png) |

|  |  |
| -- | -- |
| **COMBINED flat, V-band** | **COMBINED flat, B-band** |
| ![Combined flat, V-band](media/flat-V-combined-2026-08-22.png) | ![Combined flat, B-band](media/flat-B-combined-2026-08-22.png) |

## Image background simulator

The cell below will make an image simulator. You should use it to answer these questions:

1. Set all sliders to their lowest value. Check/unchek the "Hot pixels" box to generate images. Do they change?
2. Now increase the noise level to 10. Check/unchek the "Hot pixels" box to generate images. Do the images change? Why do you think that happens?
3. Now turn up the dark current as high as it can go, and set the noise back to zero. Check/unchek the "Hot pixels" box to generate images. Do the images change? Why do you think that happens?

In [ ]:
from ipywidgets import interactive, interact
import numpy as np

from convenience_functions import show_image
from image_sim import (read_noise, bias, dark_current, sky_background,
                       stars, sensitivity_variations)


def complete_image(bias_level=1100, read=10.0, gain=1, dark=0.1, 
                   exposure=30, hot_pixels=True, sky_counts=300,
                   n_stars=30, star_counts=2000,
                   vignetting=True, dust=True, flat_strength=1):
    synthetic_image = np.zeros([500, 500])

    # Light from the sky and stars gets to the camera *through* the
    # telescope and filter, so it is multiplied by the flat (sensitivity).
    # Bias, dark current and read noise come from the camera itself, so
    # they are NOT affected by the flat.
    flat = sensitivity_variations(synthetic_image,
                                  vignetting=vignetting, dust=dust)
    # flat_strength exaggerates the deviations from 1 so the dust donuts
    # and vignetting stand out more (1 = the simulated flat as-is).
    flat = 1 - flat_strength * (1 - flat)
    light = (sky_background(synthetic_image, sky_counts, gain=gain) +
             # fwhm here is really the Gaussian sigma; keep the stars compact
             # so they do not dominate the display stretch.
             stars(synthetic_image, n_stars, max_counts=star_counts,
                   gain=gain, fwhm=2))

    show_image(read_noise(synthetic_image, read, gain=gain) +
               bias(synthetic_image, bias_level, realistic=True) + 
               dark_current(synthetic_image, dark, exposure, gain=gain,
                            hot_pixels=hot_pixels) +
               flat * light,
               cmap='gray',
               figsize=None)
    
i = interactive(complete_image, bias_level=(1000,1200,10), dark=(0.0,1,0.1), sky_counts=(0, 1000, 50),
          gain=(0.5, 3.0, 0.25), read=(0, 50, 5.0),
          exposure=(0, 300, 30),
          n_stars=(0, 200, 10), star_counts=(500, 20000, 500),
          flat_strength=(1, 5, 1))

for kid in i.children:
    try:
        kid.continuous_update = False
    except KeyError:
        pass
i


## Questions to answer using the image simulator (discuss with others)

1. Set the "flat level" to 1. Describe how the image changes as you change the sky background.
2. Are the dust shadows still there even when they are not obvious in the images?
3. Turn up the "star counts" (i.e. star brightness) as high as it will go. Do you see the dust shadows? Are they still there?
4. When during the month will the sky background be largest? Why?